In [14]:
from dotenv import load_dotenv

load_dotenv()

True

In [15]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favorite_colour: str

# Write to state

In [16]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [17]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="ollama:qwen2.5:3b",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [18]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is Red")]},
    {"configurable": {"thread_id": "1"}}
)

In [19]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='My favourite colour is Red', additional_kwargs={}, response_metadata={}, id='428bd6ee-54b8-48fd-865b-d4fa26881e75'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-14T17:59:07.564645Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1323849875, 'load_duration': 152809917, 'prompt_eval_count': 165, 'prompt_eval_duration': 164797000, 'eval_count': 24, 'eval_duration': 989525000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f61c8-52fe-7563-bcdb-2d6aa82a1e21-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'Red'}, 'id': 'e7ebada5-d0ab-450f-a93e-2fd8f5a24f8c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 165, 'output_tokens': 24, 'total_tokens': 189}),
              ToolMessage(content='Successfully updated favourite colour', name='update_favourite_colour', id='9a2

In [20]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    }, # type: ignore
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='456298f8-ebf2-4f87-91be-7c7f55f09e39'),
              AIMessage(content="I'm just a program assistant, so I don't have feelings. How can I assist you today?", additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-14T17:59:09.62419Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1259820500, 'load_duration': 144633833, 'prompt_eval_count': 166, 'prompt_eval_duration': 153561000, 'eval_count': 22, 'eval_duration': 956451000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f61c8-5b4b-7111-b40a-eecaaed4d869-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 166, 'output_tokens': 22, 'total_tokens': 188})]}


# Read State

In [21]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    model="ollama:qwen2.5:3b",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [ ]:

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is Red")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='95a9b7a6-6852-4554-9dc2-a1d11ba2730f'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-14T17:59:11.189689Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1473870334, 'load_duration': 137726584, 'prompt_eval_count': 202, 'prompt_eval_duration': 386335000, 'eval_count': 24, 'eval_duration': 944697000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f61c8-6092-7710-9abb-61da7a57e610-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'f2319999-0958-49a0-b793-b83a5d79547c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 202, 'output_tokens': 24, 'total_tokens': 226}),
              ToolMessage(content='Successfully updated favourite colour', name='update_favourite_colour', id=

In [23]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='95a9b7a6-6852-4554-9dc2-a1d11ba2730f'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-14T17:59:11.189689Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1473870334, 'load_duration': 137726584, 'prompt_eval_count': 202, 'prompt_eval_duration': 386335000, 'eval_count': 24, 'eval_duration': 944697000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f61c8-6092-7710-9abb-61da7a57e610-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'f2319999-0958-49a0-b793-b83a5d79547c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 202, 'output_tokens': 24, 'total_tokens': 226}),
              ToolMessage(content='Successfully updated favourite colour', name='update_favourite_colour', id=

In [24]:
print(response["messages"][-1].content)

It seems like there was an issue retrieving your favourite colour. Let me know your favourite color again so I can update it for you. If you're sure that your favorite color is green, please tell me and we'll proceed.
